# Landing — ingestão bruta (Supabase → MinIO)

Issue #38 · épico #15. Lê as 10 tabelas do schema `source` no Supabase via
Spark JDBC e grava **sem transformação** como CSV no Data Lake (MinIO), uma
pasta por tabela particionada por data de ingestão. É a *fonte da verdade*
para reprocessamento das camadas seguintes.

Padrão de path: `s3a://<bucket>/landing/olist/<tabela>/ingestion_date=<run_date>/`

In [ ]:
import os

# Parâmetros (sobrescrevíveis pelo Papermill com -p)
run_date = "2026-06-17"

source_host = os.environ.get("SOURCE_DB_HOST", "")
source_port = os.environ.get("SOURCE_DB_PORT", "5432")
source_db = os.environ.get("SOURCE_DB_NAME", "postgres")
source_user = os.environ.get("SOURCE_DB_USER", "postgres")
source_password = os.environ.get("SOURCE_DB_PASSWORD", "")
source_schema = os.environ.get("SOURCE_DB_SCHEMA", "source")

minio_endpoint = os.environ.get("MINIO_ENDPOINT", "http://minio:9000")
minio_access_key = os.environ.get("MINIO_ROOT_USER", "minioadmin")
minio_secret_key = os.environ.get("MINIO_ROOT_PASSWORD", "minioadmin")
bucket = os.environ.get("DATALAKE_BUCKET", "datalake")

tables = ["categories", "customers", "sellers", "products", "orders", "addresses", "payments", "reviews", "shipments", "order_items"]

In [ ]:
APP_NAME = "engdb-landing-olist"
# Landing precisa do driver JDBC do Postgres além do conector S3A.
EXTRA_PACKAGES = ["org.apache.hadoop:hadoop-aws:3.3.4", "org.postgresql:postgresql:42.7.4"]

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder.appName(APP_NAME)
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.hadoop.fs.s3a.endpoint", minio_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", minio_access_key)
    .config("spark.hadoop.fs.s3a.secret.key", minio_secret_key)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
)
spark = configure_spark_with_delta_pip(builder, extra_packages=EXTRA_PACKAGES).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

In [ ]:
# Ingestão tabela a tabela: lê via JDBC e grava CSV cru no MinIO.
jdbc_url = f"jdbc:postgresql://{source_host}:{source_port}/{source_db}?sslmode=require"
props = {"user": source_user, "password": source_password, "driver": "org.postgresql.Driver"}

for t in tables:
    df = spark.read.jdbc(url=jdbc_url, table=f"{source_schema}.{t}", properties=props)
    dest = f"s3a://{bucket}/landing/olist/{t}/ingestion_date={run_date}"
    (df.write.mode("overwrite")
        .option("header", "true")
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
        .csv(dest))
    print(f"[landing] {t:13} linhas={df.count():>7}  ->  {dest}")

print("Landing concluída.")
spark.stop()